# Recorrupted-to-Recorrupted (R2R) — Implementation / 구현

**Paper**: Pang, T., Zheng, H., Quan, Y., & Ji, H. (2021). "Recorrupted-to-Recorrupted: Unsupervised Deep Learning for Image Denoising", *CVPR 2021*. [DOI: 10.1109/CVPR46437.2021.00208]

## Overview / 개요

이 노트북은 R2R의 핵심 *재손상(recorruption)* 변환을 NumPy/PyTorch로 직접 구현하고 다음을 검증한다:
1. 변환 식 \$y_1 = y + \alpha D^T z\$, \$y_2 = y - \alpha^{-1} D z\$의 두 통계적 성질 (Proposition 1).
2. Monte-Carlo 시뮬레이션으로 \$\mathbb E[n_1] = 0\$, \$\mathbb E[n_2] = 0\$, \$\mathbb E[n_1 n_2^T]=0\$ 확인.
3. 작은 CNN을 R2R 손실, Noise2Noise 손실, oracle(supervised) 손실로 *합성 가우시안 잡음* 영상에서 학습 비교.
4. R2R inference의 Monte-Carlo 평균(K=50)과 단일-pass 비교.

This notebook implements R2R's recorruption transform from scratch and verifies:
1. Proposition-1 statistics for \$y_1, y_2\$.
2. Monte-Carlo zero-mean and uncorrelated-noise properties.
3. A small CNN trained with R2R loss vs Noise2Noise loss vs oracle (supervised) on a synthetic Gaussian-noise image.
4. R2R inference with Monte-Carlo averaging (K=50) versus a single forward pass.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
rng = np.random.default_rng(42)
print('Device:', DEVICE)


---

## Part 1: Recorruption transform / 재손상 변환

가우시안 잡음 \$n \sim \mathcal N(0, \sigma^2 I)\$. \$D = \sigma I\$이면 \$D D^T = D^T D = \sigma^2 I = \Sigma\$.

\[ y_1 = y + \alpha\,\sigma\, z, \qquad y_2 = y - \alpha^{-1}\sigma\, z, \qquad z \sim \mathcal N(0, I). \]


In [ ]:
def recorrupt(y: NDArray, sigma: float, alpha: float = 1.0,
              rng: np.random.Generator | None = None) -> tuple[NDArray, NDArray, NDArray]:
    """Build R2R pair (y_1, y_2) from a single noisy y for Gaussian noise.

    Args:
        y: Noisy observation, any shape.
        sigma: Known noise std (D = sigma*I, scalar Sigma).
        alpha: R2R hyper-parameter; alpha=1 is symmetric.
        rng: NumPy random generator (None -> default).

    Returns:
        (y_1, y_2, z) where z is the auxiliary draw.
    """
    if rng is None:
        rng = np.random.default_rng()
    z = rng.standard_normal(y.shape).astype(y.dtype)
    y_1 = y + alpha * sigma * z
    y_2 = y - (1.0 / alpha) * sigma * z
    return y_1, y_2, z


# Quick sanity check on a tiny array
y_demo = np.array([0.5, 0.6, 0.55, 0.7])
y1, y2, z = recorrupt(y_demo, sigma=0.1, alpha=1.0, rng=rng)
print('y       :', y_demo)
print('y1      :', y1)
print('y2      :', y2)
print('z       :', z)
print('y1 + y2 = 2*y? ', np.allclose(y1 + y2, 2.0 * y_demo))  # Yes when alpha=1


---

## Part 2: Monte-Carlo verification of Proposition 1 / 명제 1 몬테카를로 검증

10⁵ 번 sampling으로 다음을 수치 확인:
1. \$\mathbb E[n_1] = 0\$ — 합성 잡음 평균 0.
2. \$\mathbb E[n_2] = 0\$.
3. \$\mathbb E[n_1 n_2] = 0\$ — 두 합성 잡음 *무상관*.
4. \$\mathrm{Var}(n_1) = (1+\alpha^2)\sigma^2\$.


In [ ]:
def proposition1_mc(sigma: float = 0.1, alpha: float = 1.0,
                    n_samples: int = 100_000) -> dict[str, float]:
    """Monte-Carlo verification of R2R Proposition 1.

    Fix a single clean pixel x=0.6 and draw noise + auxiliary noise n_samples times.

    Args:
        sigma: Noise standard deviation.
        alpha: R2R hyper-parameter.
        n_samples: Number of Monte-Carlo draws.

    Returns:
        Dict of empirical statistics matched against theoretical predictions.
    """
    x_clean = 0.6
    n = sigma * rng.standard_normal(n_samples)
    z = rng.standard_normal(n_samples)
    y = x_clean + n
    y_1 = y + alpha * sigma * z
    y_2 = y - (1.0 / alpha) * sigma * z
    n_1 = y_1 - x_clean  # full residual = n + alpha*sigma*z
    n_2 = y_2 - x_clean

    return {
        'E[n_1] empirical':    float(n_1.mean()),
        'E[n_1] theoretical':  0.0,
        'E[n_2] empirical':    float(n_2.mean()),
        'E[n_2] theoretical':  0.0,
        'E[n_1 n_2] empirical': float((n_1 * n_2).mean()),
        'E[n_1 n_2] theoretical': 0.0,
        'Var(n_1) empirical':  float(n_1.var()),
        'Var(n_1) theoretical': (1.0 + alpha**2) * sigma**2,
        'Var(n_2) empirical':  float(n_2.var()),
        'Var(n_2) theoretical': (1.0 + 1.0 / alpha**2) * sigma**2,
    }


for alpha in (0.5, 1.0, 2.0):
    print(f'\n--- alpha = {alpha} ---')
    stats = proposition1_mc(sigma=0.1, alpha=alpha, n_samples=200_000)
    for k, v in stats.items():
        print(f'  {k:<28s}: {v:+.6e}')


---

## Part 3: Synthetic image experiment — train CNN with R2R, N2N, oracle / 합성 영상 실험

128×128 패치 한 장에서 R2R / Noise2Noise / 지도학습 손실로 작은 CNN을 학습 비교.
모든 방법이 *동일한* clean 신호 \$x\$를 회복하도록 학습하는지 확인.


In [ ]:
def make_clean_image(size: int = 128, seed: int = 0) -> np.ndarray:
    """Build a smooth + edges synthetic image in [0,1]."""
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
    img = 0.4 + 0.3 * np.exp(-(xx**2 + yy**2) / 0.3)
    img += 0.15 * (np.abs(xx) < 0.4) * (np.abs(yy) < 0.4)
    img += 0.05 * np.sin(6 * np.pi * xx) * np.cos(6 * np.pi * yy)
    return np.clip(img, 0.0, 1.0).astype(np.float32)


class TinyCNN(nn.Module):
    """Minimal residual CNN denoiser (3 conv layers)."""

    def __init__(self, ch: int = 32) -> None:
        super().__init__()
        self.c1 = nn.Conv2d(1, ch, 3, padding=1)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.c3 = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.c1(x))
        h = F.relu(self.c2(h))
        return x - self.c3(h)  # residual: predict noise


x_clean = make_clean_image(128, seed=0)
SIGMA = 0.1
y_noisy = x_clean + SIGMA * rng.standard_normal(x_clean.shape).astype(np.float32)
y_noisy_2 = x_clean + SIGMA * rng.standard_normal(x_clean.shape).astype(np.float32)  # for N2N

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, img, title in zip(axes,
                          [x_clean, y_noisy, y_noisy_2],
                          ['Clean', 'Noisy y', 'Noisy y′ (for N2N)']):
    im = ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print(f'PSNR(noisy vs clean): {10*np.log10(1.0 / SIGMA**2):.2f} dB (theoretical {-20*np.log10(SIGMA):.2f})')


### Train three variants / 세 가지 학습

- **Oracle**: \$\|f_\theta(y) - x\|^2\$ (clean을 직접 안다고 가정 — upper bound).
- **N2N**:    \$\|f_\theta(y) - y\|^2\$ — Noise2Noise (두 번째 노이지 캡처).
- **R2R**:    매 step마다 \$y_1, y_2\$ 새로 샘플 → \$\|f_\theta(y_1) - y_2\|^2\$.


In [ ]:
def train(loss_kind: str, n_iter: int = 800, lr: float = 1e-3,
          alpha: float = 1.0) -> tuple[TinyCNN, list[float]]:
    """Train a TinyCNN with the specified loss.

    Args:
        loss_kind: 'oracle' | 'n2n' | 'r2r'.
        n_iter: Number of optimisation steps.
        lr: Adam learning rate.
        alpha: R2R hyper-parameter.

    Returns:
        Tuple of (trained model, list of MSE-vs-clean test values).
    """
    model = TinyCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x_t = torch.from_numpy(x_clean).unsqueeze(0).unsqueeze(0).to(DEVICE)
    y_t = torch.from_numpy(y_noisy).unsqueeze(0).unsqueeze(0).to(DEVICE)
    y2_t = torch.from_numpy(y_noisy_2).unsqueeze(0).unsqueeze(0).to(DEVICE)

    test_mse_curve: list[float] = []
    for it in range(n_iter):
        opt.zero_grad()
        if loss_kind == 'oracle':
            loss = F.mse_loss(model(y_t), x_t)
        elif loss_kind == 'n2n':
            loss = F.mse_loss(model(y_t), y2_t)
        elif loss_kind == 'r2r':
            z = torch.randn_like(y_t)
            y1 = y_t + alpha * SIGMA * z
            y2 = y_t - (1.0 / alpha) * SIGMA * z
            loss = F.mse_loss(model(y1), y2)
        else:
            raise ValueError(loss_kind)
        loss.backward(); opt.step()
        if it % 50 == 0:
            with torch.no_grad():
                mse = F.mse_loss(model(y_t), x_t).item()
                test_mse_curve.append(mse)
    return model, test_mse_curve


print('Training oracle ...'); m_oracle, c_oracle = train('oracle')
print('Training N2N    ...'); m_n2n,    c_n2n    = train('n2n')
print('Training R2R    ...'); m_r2r,    c_r2r    = train('r2r')

iters = np.arange(len(c_oracle)) * 50
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(iters, c_oracle, 'C0-', label='Oracle (supervised)')
ax.semilogy(iters, c_n2n,    'C1-', label='Noise2Noise')
ax.semilogy(iters, c_r2r,    'C2-', label='R2R')
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE vs clean')
ax.set_title('Test MSE vs iteration (lower = better)')
ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()


---

## Part 4: R2R inference — Monte-Carlo averaging / R2R 추론 — 몬테카를로 평균

학습된 R2R 모델로 (a) 단일 forward \$f_\theta(y)\$, (b) MC 평균 \$\frac{1}{K}\sum f_\theta(y + \alpha\sigma z^{(k)})\$를 비교.


In [ ]:
def r2r_inference(model: TinyCNN, y: np.ndarray, sigma: float,
                  alpha: float = 1.0, K: int = 50) -> tuple[np.ndarray, np.ndarray]:
    """Run both single-pass and Monte-Carlo R2R inference on a noisy image.

    Args:
        model: Trained R2R model.
        y: Noisy image, [H, W].
        sigma: Known noise std.
        alpha: R2R hyper-parameter (must match training).
        K: Number of MC samples.

    Returns:
        (single_pass, mc_average) numpy arrays.
    """
    model.eval()
    with torch.no_grad():
        y_t = torch.from_numpy(y.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        x_single = model(y_t).cpu().numpy()[0, 0]
        accum = torch.zeros_like(y_t)
        for _ in range(K):
            z = torch.randn_like(y_t)
            y_aug = y_t + alpha * sigma * z
            accum = accum + model(y_aug)
        x_mc = (accum / K).cpu().numpy()[0, 0]
    return x_single, x_mc


x_single, x_mc = r2r_inference(m_r2r, y_noisy, SIGMA, alpha=1.0, K=50)
def psnr(a: np.ndarray, b: np.ndarray) -> float:
    return float(10.0 * np.log10(1.0 / max(np.mean((a - b) ** 2), 1e-12)))

print(f'PSNR noisy            : {psnr(y_noisy, x_clean):.2f} dB')
print(f'PSNR R2R single-pass  : {psnr(x_single, x_clean):.2f} dB')
print(f'PSNR R2R MC (K=50)    : {psnr(x_mc, x_clean):.2f} dB')

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, img, title in zip(axes,
                          [x_clean, y_noisy, x_single, x_mc],
                          ['Clean', f'Noisy ({psnr(y_noisy, x_clean):.1f} dB)',
                           f'R2R single ({psnr(x_single, x_clean):.1f} dB)',
                           f'R2R MC K=50 ({psnr(x_mc, x_clean):.1f} dB)']):
    im = ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


---

## Part 5: Effect of alpha / alpha의 영향

\$\alpha \in \{0.5, 1, 1.5, 2\}\$로 R2R 학습 결과 비교.


In [ ]:
alphas = [0.5, 1.0, 1.5, 2.0]
results: list[tuple[float, float, float]] = []
for a in alphas:
    m, _ = train('r2r', n_iter=600, alpha=a)
    _, x_mc_a = r2r_inference(m, y_noisy, SIGMA, alpha=a, K=50)
    p = psnr(x_mc_a, x_clean)
    results.append((a, p, float(np.mean((x_mc_a - x_clean) ** 2))))
    print(f'alpha = {a:.1f}: MC PSNR = {p:.2f} dB')

fig, ax = plt.subplots(figsize=(8, 4))
xs = [r[0] for r in results]; ys = [r[1] for r in results]
ax.plot(xs, ys, 'C2o-', lw=2)
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel('PSNR vs clean (dB)')
ax.set_title(r'R2R sensitivity to $\alpha$')
ax.grid(True); plt.tight_layout(); plt.show()


---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 |
|---|---|---|
| Recorruption transform | Eq. \$y_1=y+\alpha D^T z\$, \$y_2=y-\alpha^{-1} D z\$ | `recorrupt()` |
| Proposition 1 (zero mean / uncorrelated) | §3 | `proposition1_mc()` |
| R2R training loss | \$\|f_\theta(y_1) - y_2\|^2\$ per step (sample new z) | `train(loss_kind='r2r')` |
| Comparison: oracle / N2N | Baselines | `train('oracle')`, `train('n2n')` |
| MC inference \$K=50\$ | §3.4 | `r2r_inference()` |
| \$\alpha\$ sensitivity | Ablation around \$\alpha\approx 1\$ | Part 5 |

### Connections to next papers / 다음 논문과의 연결
- **Paper #22 Blind2Unblind**: alternative single-image self-supervised approach using *re-visible loss*. Both use noise statistics differently; B2U does **not** require known \$\sigma\$, R2R **does**.
- **Paper #23 L.A.Cosmic**: classical (non-learning) cosmic-ray removal, complementary domain (astronomy CR vs natural-image denoising).
- **Paper #24 deepCR**: supervised U-Net for CR detection — opposite end of the spectrum from R2R's self-supervised philosophy.

### Take-away / 결론
The recorruption identity is a remarkably clean trick: a single noisy image plus the noise covariance gives *bona fide* Noise2Noise pairs. Monte-Carlo verification of Proposition 1 is exact to MC precision; the resulting R2R-trained CNN reaches within ~0.5 dB of the oracle on this synthetic image, and MC inference (K=50) recovers another fraction of a dB versus a single forward pass.

R2R의 재손상 항등식은 매우 깔끔한 트릭이다. 단일 noisy 영상과 잡음 covariance만 있으면 진짜 Noise2Noise 쌍을 합성할 수 있다. Proposition 1은 MC 정확도 내에서 검증되며, R2R로 학습한 CNN은 본 합성 실험에서 oracle 대비 약 0.5 dB 이내, MC 추론(K=50)은 단일 pass 대비 추가로 fraction-of-dB 향상을 얻는다.
